# CaféGuard PR — Coffee-Leaf Disease Classification & INT8 Edge Export

**Purpose:** Train a **MobileNetV3Small** coffee-leaf classifier, compare **FP32 TFLite**, **full-integer INT8 post-training quantization (PTQ)**, and **INT8 with classifier-head quantization-aware fine-tuning (head-QAT)**, then export models for Raspberry Pi live-camera inference.

**What this notebook does:** Loads coffee-leaf photographs, prepares a held-out train/validation/test split, trains a baseline, exports and evaluates three TFLite models, plots metrics, and produces a deployment ZIP.

**What it does not do:** It cannot measure actual Raspberry Pi CPU, RAM, sustained camera FPS, or power from a Colab VM. Run the included `cafeguard_pi_camera.py` on the Pi and report those numbers separately. Head-QAT is **not** whole-network QAT.

**Dataset choices:**

- **Automatic runnable demonstration:** Hugging Face `Project-AgML/arabica_coffee_leaf_disease_classification` (JMuBEN/JMuBEN2-derived). The complete download is approximately **1.84 GB**. Selects **Healthy** and **Leaf_rust** only, deduplicates exact decoded images, and caps the selection for quick experiments. **The original dataset has many duplicates; do not claim this quickstart as your final evaluation.**
- **Your planned RoCoLe study:** Set `DATA_SOURCE = 'upload_zip'` in the config cell and upload a ZIP with folders `healthy/` and `rust/` (rust levels 1–4 can have separate `rust_level_*` folders). Mite cases are excluded rather than being mislabeled as rust. Or provide an explicit `labels.csv` with `filename,label` columns. This is the preferred route for the project once images are prepared.

The **RoCoLe** source has 1,560 photos, but this experiment's two-class task is *healthy versus rust*; mite examples should not be forced into either class. Dataset origin is **Ecuador** (not Puerto Rico). Applicability to Puerto Rican farms will require field testing.

**Sources:** [RoCoLe](https://data.mendeley.com/datasets/c5yvn32dzg/2) • [AgML Arabica coffee dataset](https://huggingface.co/datasets/Project-AgML/arabica_coffee_leaf_disease_classification) • [TensorFlow quantization guide](https://www.tensorflow.org/model_optimization/guide/quantization/training_comprehensive_guide)

## 1. Install dependencies

Use **Runtime → Change runtime type → T4 GPU** if available, then run cells in order. The first pip cell installs the compatible legacy-Keras stack used by TensorFlow Model Optimization. **Restart the runtime after installation if Colab requests it**, then resume at the next cell. The setup relies on Internet access in Colab to download the dataset and ImageNet backbone weights.

In [ ]:
!python3 --version

In [ ]:
!pip install tensorflow==2.20.0

In [ ]:
%pip -q install "tensorflow==2.20.0" "tf-keras==2.20.1" "tensorflow-model-optimization==0.8.1" "datasets>=3.0,<5" "scikit-learn>=1.4" "pillow" "pandas" "matplotlib" "seaborn" "nbformat"

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'  # MUST precede TensorFlow import
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import json, time, hashlib, random, shutil, zipfile, io
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import tensorflow as tf
import tf_keras as keras
import tensorflow_model_optimization as tfmot
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| Legacy Keras', keras.__version__, '| TFMOT', tfmot.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Configure project

Default **Hugging Face** mode automatically acquires actual coffee-leaf images. For the intended **RoCoLe** experiment, switch to `upload_zip` and supply prepared labeled images (ZIP root may have a surrounding folder). Uploaded images must already be labeled correctly. Aim for a group-aware split by plant/leaf ID if your source provides them.

In [ ]:
DATA_SOURCE = 'huggingface'  # 'huggingface' or 'upload_zip'
MAX_PER_CLASS = 250         # quickstart cap; increase for experiments
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 5
EPOCHS_FINE_TUNE = 2
EPOCHS_HEAD_QAT = 3
TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION = 0.64, 0.16, 0.20
OUTPUT_DIR = Path('/content/cafeguard_outputs')
DATA_DIR = Path('/content/cafeguard_images')
CLASS_NAMES = ['healthy', 'rust']  # fixed ordering, 0=healthy, 1=rust
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not np.isclose(TRAIN_FRACTION+VAL_FRACTION+TEST_FRACTION, 1):
    raise ValueError('Split fractions must sum to 1')

## 3. Load and label images

**Hugging Face:** approximately 1.84 GB to download; this is a research quickstart rather than the final RoCoLe benchmark. Exact duplicates are filtered **before** train/test splitting. Near-duplicate photos of the same leaf may still exist and should be grouped before publication-level evaluations.

**Upload ZIP:** Accepted structures include `healthy/*.jpg`, `rust/*.jpg`, `rust_level_1/*.jpg`, ..., `rust_level_4/*.jpg`, and `red_spider_mite/*.jpg` (excluded). Alternatively include a `labels.csv` with `filename` (relative path or filename) and `label` (`healthy` / `rust_level_1` / etc.).

In [ ]:
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
for label in CLASS_NAMES:
    (DATA_DIR / label).mkdir(parents=True, exist_ok=True)

def binary_label(raw):
    s = str(raw).strip().lower().replace('-', '_').replace(' ', '_')
    if 'unhealthy' in s or 'mite' in s or 'miner' in s or 'phoma' in s or 'cercospora' in s:
        return None
    if s in ('healthy','health','0_healthy') or s.startswith('healthy_'):
        return 'healthy'
    if 'rust' in s or s == 'clr':
        return 'rust'
    return None

counts = Counter(); seen = set(); accepted = []
def ingest_pil(pil_image, label, source_name='image'):
    mapped = binary_label(label)
    if mapped is None or counts[mapped] >= MAX_PER_CLASS:
        return False
    try:
        im = ImageOps.exif_transpose(pil_image).convert('RGB')
        im = ImageOps.fit(im, IMAGE_SIZE, method=Image.Resampling.BILINEAR)
        digest = hashlib.sha256(im.tobytes()).hexdigest()
        if digest in seen:
            return False
        seen.add(digest)
        dest = DATA_DIR / mapped / f'{digest[:24]}.jpg'
        im.save(dest, quality=94)
        counts[mapped] += 1
        accepted.append((str(dest), CLASS_NAMES.index(mapped), source_name))
        return True
    except Exception as exc:
        print('Skipping unreadable image:', source_name, exc)
        return False

if DATA_SOURCE == 'huggingface':
    from datasets import load_dataset
    hf_id = 'Project-AgML/arabica_coffee_leaf_disease_classification'
    ds = load_dataset(hf_id, split='train')
    print('Downloaded', len(ds), 'rows from', hf_id, '| columns:', ds.column_names)
    feat = ds.features.get('label')
    raw_names = getattr(feat, 'names', None)
    if not {'image', 'label'}.issubset(ds.column_names):
        raise ValueError('Unexpected Hugging Face schema: expected image and label')
    # Iterate in randomized order, stopping when both classes reach the cap.
    for item in ds.shuffle(seed=SEED):
        raw_label = item['label']
        if isinstance(raw_label, (int, np.integer)) and raw_names:
            raw_label = raw_names[int(raw_label)]
        mapped = binary_label(raw_label)
        if mapped is None or counts[mapped] >= MAX_PER_CLASS:
            continue
        ingest_pil(item['image'], mapped, 'AgML/JMuBEN')
        if all(counts[c] >= MAX_PER_CLASS for c in CLASS_NAMES):
            break
elif DATA_SOURCE == 'upload_zip':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
    if not zip_names:
        raise ValueError('Please upload a ZIP of labeled coffee-leaf images.')
    extract_dir = Path('/content/cafeguard_uploaded')
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    extract_dir.mkdir()
    for zip_name in zip_names:
        with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
            for member in z.infolist():
                p = Path(member.filename)
                if member.is_dir() or p.is_absolute() or '..' in p.parts:
                    continue
                dst = extract_dir.joinpath(*p.parts)
                dst.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, open(dst, 'wb') as target:
                    shutil.copyfileobj(src, target)
    images = [p for p in extract_dir.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    manifest_paths = [p for p in extract_dir.rglob('labels.csv')]
    manifest = {}
    if manifest_paths:
        df_manifest = pd.read_csv(manifest_paths[0])
        if not {'filename', 'label'}.issubset(df_manifest.columns):
            raise ValueError('labels.csv must contain filename,label columns')
        for row in df_manifest.itertuples(index=False):
            manifest[str(getattr(row, 'filename')).replace('\\','/').lower()] = getattr(row, 'label')
    for p in sorted(images):
        rel = p.relative_to(extract_dir).as_posix().lower()
        annotation = manifest.get(rel, manifest.get(p.name.lower()))
        if annotation is None:
            annotation = next((part for part in reversed(p.parent.relative_to(extract_dir).parts) if binary_label(part)), None)
        mapped = binary_label(annotation) if annotation is not None else None
        if mapped and counts[mapped] < MAX_PER_CLASS:
            with Image.open(p) as im:
                ingest_pil(im, mapped, str(p))
else:
    raise ValueError('DATA_SOURCE must be huggingface or upload_zip')

print('Selected, exact-deduplicated images:', dict(counts))
if min(counts.get(c, 0) for c in CLASS_NAMES) < 20:
    raise ValueError('Need >=20 distinct photos in each of healthy and rust. Check labeling/source.')
source_description = 'AgML/JMuBEN HF quickstart' if DATA_SOURCE=='huggingface' else 'Uploaded labeled coffee-leaf ZIP'

## 4. Examine examples and split data without leakage from exact duplicates

The exact image hash was deduplicated above before the split. **Important:** RoCoLe photographs include repeated views from the same plants. If plant IDs are available, use a **plant-grouped split** before reporting final results; this random image split is only a preliminary experiment.

In [ ]:
paths = np.array([p for p, _, _ in accepted])
labels = np.array([y for _, y, _ in accepted], dtype=np.int64)
indices = np.arange(len(paths))
train_val_idx, test_idx = train_test_split(indices, test_size=TEST_FRACTION, random_state=SEED, stratify=labels)
train_idx, val_idx = train_test_split(train_val_idx, test_size=VAL_FRACTION/(TRAIN_FRACTION+VAL_FRACTION), random_state=SEED, stratify=labels[train_val_idx])
print('Train',len(train_idx),'Val',len(val_idx),'Test',len(test_idx))
for name, idx in [('train',train_idx),('validation',val_idx),('test',test_idx)]:
    print(name, dict(Counter(labels[idx])))

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axs.flat, np.random.default_rng(SEED).choice(len(paths), size=8, replace=False)):
    ax.imshow(Image.open(paths[i])); ax.set_title(CLASS_NAMES[int(labels[i])]); ax.axis('off')
plt.tight_layout(); plt.show()

pd.DataFrame({'path':paths, 'label':labels}).assign(split=[('train' if j in set(train_idx) else 'validation' if j in set(val_idx) else 'test') for j in indices]).to_csv(OUTPUT_DIR/'dataset_split.csv',index=False)

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()

with zipfile.ZipFile("CafeGuard_PR_Deployment.zip") as z:
    z.extractall(OUTPUT_DIR)

MODEL_PATHS = {
    "FP32": OUTPUT_DIR / "cafeguard_fp32.tflite",
    "INT8_PTQ": OUTPUT_DIR / "cafeguard_int8_ptq.tflite",
    "INT8_HEAD_QAT": OUTPUT_DIR / "cafeguard_int8_head_qat.tflite"
}

for name, path in MODEL_PATHS.items():
    print(name, path.exists())

In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

def load_raw(path):
    with Image.open(path) as im:
        return np.asarray(
            ImageOps.fit(
                im.convert("RGB"),
                IMAGE_SIZE,
                method=Image.Resampling.BILINEAR
            ),
            dtype=np.float32
        )

def tflite_logits(interpreter, image):
    x_info = interpreter.get_input_details()[0]
    y_info = interpreter.get_output_details()[0]

    x = np.expand_dims(image, axis=0)

    if np.issubdtype(x_info["dtype"], np.integer):
        scale, zp = x_info["quantization"]
        limits = np.iinfo(x_info["dtype"])

        x = np.clip(
            np.round(x / scale + zp),
            limits.min,
            limits.max
        ).astype(x_info["dtype"])
    else:
        x = x.astype(x_info["dtype"])

    interpreter.set_tensor(x_info["index"], x)
    interpreter.invoke()

    out = interpreter.get_tensor(y_info["index"])[0]

    if np.issubdtype(y_info["dtype"], np.integer):
        scale, zp = y_info["quantization"]
        out = (out.astype(np.float32) - zp) * scale

    return out

results = []

print("Test images:", len(test_idx))

for name, path in MODEL_PATHS.items():

    print("\nEvaluating:", name)

    interpreter = tf.lite.Interpreter(
        model_path=str(path),
        num_threads=2,
        experimental_op_resolver_type=(
            tf.lite.experimental.OpResolverType
            .BUILTIN_WITHOUT_DEFAULT_DELEGATES
        )
    )

    interpreter.allocate_tensors()

    test_y = labels[test_idx]
    predictions = []

    for idx in test_idx:
        image = load_raw(paths[idx])
        output = tflite_logits(interpreter, image)
        predictions.append(int(np.argmax(output)))

    accuracy = accuracy_score(test_y, predictions)

    p, r, f, _ = precision_recall_fscore_support(
        test_y,
        predictions,
        labels=[0, 1],
        average="macro",
        zero_division=0
    )

    results.append({
        "model": name,
        "test_images": len(test_y),
        "correct_predictions": int(
            np.sum(test_y == np.array(predictions))
        ),
        "accuracy_percent": accuracy * 100,
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f,
        "size_mib": path.stat().st_size / 1048576
    })

    print("Accuracy:", accuracy * 100, "%")

    print(classification_report(
        test_y,
        predictions,
        labels=[0, 1],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(
        test_y, predictions, labels=[0, 1]
    ))

    del interpreter

accuracy_results = pd.DataFrame(results)

accuracy_results.to_csv(
    "/content/final_accuracy_results.csv",
    index=False
)

display(accuracy_results.round(4))


In [ ]:
from google.colab import files
files.download("/content/final_accuracy_results.csv")

## 5. TensorFlow data pipeline and reproducible augmentation

**Input convention:** raw **RGB** `float32` pixels in **[0,255]**. `MobileNetV3Small(include_preprocessing=True)` does its own normalization. The same raw-pixel convention is used for TFLite and Pi camera inputs.

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE

def _decode(path, y):
    raw=tf.io.read_file(path)
    image=tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None,None,3])
    image=tf.image.resize(tf.cast(image,tf.float32), IMAGE_SIZE, method='bilinear')
    return image, tf.cast(y,tf.int32)

def _augment(image, y):
    image=tf.image.random_flip_left_right(image)
    image=tf.image.random_flip_up_down(image)
    image=tf.image.random_brightness(image, 14.0)
    image=tf.image.random_contrast(image,0.9,1.1)
    return tf.clip_by_value(image,0.,255.), y

def make_ds(ix, training=False):
    d=tf.data.Dataset.from_tensor_slices((paths[ix],labels[ix]))
    if training: d=d.shuffle(max(128, len(ix)),seed=SEED,reshuffle_each_iteration=True)
    d=d.map(_decode,num_parallel_calls=AUTOTUNE)
    if training: d=d.map(_augment,num_parallel_calls=AUTOTUNE)
    return d.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds=make_ds(train_idx,training=True)
train_plain_ds=make_ds(train_idx,training=False)
val_ds=make_ds(val_idx)
test_ds=make_ds(test_idx)

## 6. Train FP32 MobileNetV3Small baseline

A frozen ImageNet backbone is trained with a two-class head, then the upper portion of the backbone is optionally fine-tuned at a lower learning rate.

In [ ]:
backbone=keras.applications.MobileNetV3Small(
    include_top=False,weights='imagenet',input_shape=(*IMAGE_SIZE,3),
    include_preprocessing=True
)
backbone.trainable=False
features_dim=int(backbone.output_shape[-1])
head=keras.Sequential([
    keras.layers.InputLayer(input_shape=(features_dim,)),
    keras.layers.Dense(len(CLASS_NAMES),activation='softmax',name='classifier')
],name='classification_head')

def connect_head(the_head, name):
    inp=keras.Input(shape=(*IMAGE_SIZE,3),name='rgb_raw_0_255')
    z=backbone(inp,training=False)
    z=keras.layers.GlobalAveragePooling2D(name='avg_pool')(z)
    prob=the_head(z)
    return keras.Model(inp,prob,name=name)

fp32_model=connect_head(head,'cafeguard_fp32')
fp32_model.compile(optimizer=keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
weights={i:len(labels[train_idx])/(len(CLASS_NAMES)*max(1,int(np.sum(labels[train_idx]==i)))) for i in range(len(CLASS_NAMES))}
cb=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)]
print('Class weights:',weights)
hist_head=fp32_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_HEAD,class_weight=weights,callbacks=cb,verbose=1)
if EPOCHS_FINE_TUNE > 0:
    backbone.trainable=True
    for layer in backbone.layers[:-30]: layer.trainable=False
    # Keep batch-normalization stable on small datasets.
    for layer in backbone.layers:
        if isinstance(layer, keras.layers.BatchNormalization): layer.trainable=False
    fp32_model.compile(optimizer=keras.optimizers.Adam(1e-5),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    hist_ft=fp32_model.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_FINE_TUNE,class_weight=weights,verbose=1)
else:
    hist_ft=None
# Freeze again before constructing an independent head-QAT model.
backbone.trainable=False
fp32_model.save_weights(str(OUTPUT_DIR/'fp32_weights.h5'))

## 7. INT8 classifier-head QAT

The standard TFMOT API cannot blindly wrap every MobileNetV3 layer, so this **selective QAT** path is designed to be more compatible: the **classification head** simulates quantization during fine-tuning, while the **backbone** stays frozen and is integer-quantized by TFLite conversion calibration. The resulting exported model is INT8, but its QAT training coverage is **head only**. State this precisely in your report.

In [ ]:
qat_head = tfmot.quantization.keras.quantize_model(head)
qat_model = connect_head(qat_head,'cafeguard_head_qat')
qat_model.compile(optimizer=keras.optimizers.Adam(1e-4),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
qat_history = qat_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD_QAT,
                            class_weight=weights,verbose=1)
print('QAT head trainable params:', sum(int(np.prod(w.shape)) for w in qat_head.trainable_weights))

## 8. Export FP32, INT8 PTQ, and INT8 head-QAT TFLite models

**Full integer quantization** uses a representative set of unaugmented training images. Both INT8 artifacts have **int8 inputs and outputs**; we verify the tensors and fail loudly if conversion is unsupported. No silent fallback to float is permitted.

In [ ]:
def representative_data():
    for imgs, _ in train_plain_ds.unbatch().batch(1).take(100):
        yield [tf.cast(imgs,tf.float32)]

def export_fp32(model,target):
    converter=tf.lite.TFLiteConverter.from_keras_model(model)
    target.write_bytes(converter.convert())
    return target

def export_int8(model,target):
    converter=tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations=[tf.lite.Optimize.DEFAULT]
    converter.representative_dataset=representative_data
    converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type=tf.int8
    converter.inference_output_type=tf.int8
    model_bytes=converter.convert()
    target.write_bytes(model_bytes)

    # Load interpreter without allocating tensors to check metadata and avoid XNNPACK delegate initialization errors
    interpreter=tf.lite.Interpreter(model_path=str(target))
    info_in=interpreter.get_input_details()[0]
    info_out=interpreter.get_output_details()[0]
    assert info_in['dtype']==np.int8 and info_out['dtype']==np.int8, 'Not fully INT8 I/O!'
    return target

MODEL_PATHS={
    'FP32':export_fp32(fp32_model, OUTPUT_DIR/'cafeguard_fp32.tflite'),
    'INT8_PTQ':export_int8(fp32_model, OUTPUT_DIR/'cafeguard_int8_ptq.tflite'),
    'INT8_HEAD_QAT':export_int8(qat_model, OUTPUT_DIR/'cafeguard_int8_head_qat.tflite'),
}
(OUTPUT_DIR/'labels.json').write_text(json.dumps(CLASS_NAMES,indent=2))
for n,p in MODEL_PATHS.items():
    print(n, ':', round(p.stat().st_size/1024/1024,3),'MiB')

## 9. Validate all three *exported* TFLite models

Accuracy and macro precision/recall/F1 are calculated on exactly the same held-out images. Inference time here is for **this Colab CPU**, **not** a Raspberry Pi. If a quantized result looks suspicious, inspect the TFLite input scale, zero point, RGB conversion and preprocessing.

In [ ]:
def tflite_logits(interpreter, raw_rgb_float):
    x_info=interpreter.get_input_details()[0]
    y_info=interpreter.get_output_details()[0]
    x=np.expand_dims(np.asarray(raw_rgb_float,dtype=np.float32),axis=0)
    if np.issubdtype(x_info['dtype'], np.integer):
        scale,zp=x_info['quantization']
        if not scale: raise ValueError('Missing input quantization scale')
        info=np.iinfo(x_info['dtype'])
        x=np.clip(np.round(x/scale+zp),info.min,info.max).astype(x_info['dtype'])
    else:
        x=x.astype(x_info['dtype'])
    interpreter.set_tensor(x_info['index'],x)
    interpreter.invoke()
    out=interpreter.get_tensor(y_info['index'])[0]
    if np.issubdtype(y_info['dtype'],np.integer):
        scale,zp=y_info['quantization']
        out=(out.astype(np.float32)-zp)*scale
    return np.asarray(out,dtype=np.float32)

# For low memory use, load one test image at a time.
def load_raw(path):
    with Image.open(path) as im:
        return np.asarray(ImageOps.fit(im.convert('RGB'),IMAGE_SIZE,method=Image.Resampling.BILINEAR),dtype=np.float32)

metrics=[]; predictions={}
for name, path in MODEL_PATHS.items():
    # Bypass default XNNPACK delegate by configuring empty experimental delegates and limiting threads
    try:
        interpreter = tf.lite.Interpreter(model_path=str(path), experimental_delegates=[])
        if hasattr(interpreter, '_interpreter') and interpreter._interpreter:
            try:
                interpreter._interpreter.SetNumThreads(1)
            except Exception:
                pass
        interpreter.allocate_tensors()
    except Exception:
        try:
            interpreter = tf.lite.Interpreter(model_path=str(path))
            interpreter.allocate_tensors()
        except Exception as e:
            print(f"Could not load {name}: {e}")
            continue

    test_y=labels[test_idx]
    pred=[]
    for idx in test_idx:
        prob=tflite_logits(interpreter,load_raw(paths[idx]))
        pred.append(int(np.argmax(prob)))
    predictions[name]=pred
    p,r,f,_=precision_recall_fscore_support(test_y,pred,average='macro',zero_division=0)
    # Warm-up before timing, then 100 measured single-image inference passes.
    sample=load_raw(paths[test_idx[0]])
    for _ in range(10): tflite_logits(interpreter,sample)
    times=[]
    for _ in range(100):
        t0=time.perf_counter(); tflite_logits(interpreter,sample)
        times.append((time.perf_counter()-t0)*1000)
    metrics.append({'model':name,'accuracy':accuracy_score(test_y,pred),
                    'macro_precision':p,'macro_recall':r,'macro_f1':f,
                    'size_mib':path.stat().st_size/(1024**2),
                    'colab_cpu_mean_ms':float(np.mean(times)),
                    'colab_cpu_p95_ms':float(np.percentile(times,95))})
    print('\n',name)
    print(classification_report(test_y,pred,target_names=CLASS_NAMES,zero_division=0))
    cm=confusion_matrix(test_y,pred,labels=[0,1])
    fig,ax=plt.subplots(figsize=(4,3))
    ax.imshow(cm,cmap='Blues')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,
           xlabel='Predicted',ylabel='True',title=f'{name} test confusion matrix')
    for (r0,c0),v in np.ndenumerate(cm): ax.text(c0,r0,str(v),ha='center',va='center')
    plt.tight_layout(); fig.savefig(OUTPUT_DIR/f'confusion_{name}.png',dpi=160); plt.show()

results_df=pd.DataFrame(metrics)
results_df.to_csv(OUTPUT_DIR/'colab_results.csv',index=False)
display(results_df.round(4))

In [ ]:
from pathlib import Path

print("Dataset available:", "test_idx" in globals())
print("Labels available:", "labels" in globals())
print("Evaluation function available:", "tflite_logits" in globals())

print("\nAvailable models:")
for p in Path("/content").rglob("*.tflite"):
    print(p)

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

metrics = []

for name, path in MODEL_PATHS.items():
    print("\nEvaluating:", name)

    interpreter = tf.lite.Interpreter(
        model_path=str(path),
        num_threads=2,
        experimental_op_resolver_type=(
            tf.lite.experimental.OpResolverType
            .BUILTIN_WITHOUT_DEFAULT_DELEGATES
        )
    )

    interpreter.allocate_tensors()

    test_y = labels[test_idx]
    pred = []

    for idx in test_idx:
        image = load_raw(paths[idx])
        output = tflite_logits(interpreter, image)
        pred.append(int(np.argmax(output)))

    accuracy = accuracy_score(test_y, pred)

    p, r, f, _ = precision_recall_fscore_support(
        test_y, pred,
        average="macro",
        zero_division=0
    )

    metrics.append({
        "model": name,
        "test_images": len(test_y),
        "accuracy_percent": accuracy * 100,
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f,
        "size_mib": path.stat().st_size / 1048576
    })

    print("Accuracy:", accuracy * 100, "%")

    print(classification_report(
        test_y, pred,
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    print("Confusion matrix:")
    print(confusion_matrix(test_y, pred))

    del interpreter

accuracy_results = pd.DataFrame(metrics)

accuracy_results.to_csv(
    "/content/final_accuracy_results.csv",
    index=False
)

display(accuracy_results)


## 10. Generate figures and deployment bundle

When you run on your Raspberry Pi, benchmark each model for **100+ warmed-up inferences** with the same camera, image size, lighting and thread settings. Use the included Pi script for live classification and measurement. The project's intended goals are **≥85% held-out accuracy** and **<500 ms/image**, but these are *targets*, not results.

In [ ]:
# Bundle the real-time Raspberry Pi camera inference script with the models.
PI_SCRIPT = '#!/usr/bin/env python3\n"""CaféGuard PR Raspberry Pi live camera inference + repeatable CPU benchmark.\n\nExamples:\n python3 cafeguard_pi_camera.py --model cafeguard_int8_ptq.tflite --benchmark 100 --frames 200\n python3 cafeguard_pi_camera.py --model cafeguard_fp32.tflite --camera usb --show\n\nRequires a Raspberry Pi-compatible LiteRT interpreter (ai-edge-litert or tflite-runtime),\nNumPy, Pillow, and either Picamera2 (CSI Pi camera) or OpenCV (USB camera).\n"""\nimport argparse\nimport json\nimport time\nimport csv\nfrom pathlib import Path\nfrom collections import deque\nimport numpy as np\nfrom PIL import Image, ImageOps\n\n\ndef get_interpreter(model, threads):\n    errors = []\n    for module in (\'ai_edge_litert.interpreter\', \'tflite_runtime.interpreter\'):\n        try:\n            interpreter = __import__(module, fromlist=[\'Interpreter\']).Interpreter(\n                model_path=str(model), num_threads=threads)\n            interpreter.allocate_tensors()\n            return interpreter, module\n        except (ImportError, OSError) as exc:\n            errors.append(f\'{module}: {exc}\')\n    try:\n        import tensorflow as tf\n        interpreter = tf.lite.Interpreter(model_path=str(model), num_threads=threads)\n        interpreter.allocate_tensors()\n        return interpreter, \'tensorflow.lite\'\n    except (ImportError, OSError) as exc:\n        errors.append(f\'tensorflow.lite: {exc}\')\n    raise RuntimeError(\'No compatible TFLite/LiteRT runtime found.\\n\'+\'\\n\'.join(errors))\n\n\ndef open_camera(preference):\n    if preference in (\'auto\',\'pi\'):\n        try:\n            from picamera2 import Picamera2\n            camera = Picamera2()\n            camera.configure(camera.create_preview_configuration(\n                main={\'size\': (640, 480), \'format\': \'RGB888\'}))\n            camera.start()\n            time.sleep(0.5)\n            return camera, \'pi\'\n        except (ImportError, OSError, RuntimeError) as exc:\n            if preference == \'pi\':\n                raise RuntimeError(f\'Could not start Pi camera: {exc}\') from exc\n            print(\'Pi Camera unavailable; trying USB:\', exc)\n    try:\n        import cv2\n    except ImportError as exc:\n        raise RuntimeError(\'USB camera requires OpenCV; install python3-opencv.\') from exc\n    cap = cv2.VideoCapture(0)\n    if not cap.isOpened():\n        cap.release()\n        raise RuntimeError(\'Could not open Pi Camera or USB /dev/video0\')\n    return cap, \'usb\'\n\n\ndef capture_rgb(cam, camera_type):\n    if camera_type == \'pi\':\n        rgb = cam.capture_array()\n        # The PiCamera2 RGB888 output is requested explicitly above.\n        return np.asarray(rgb)[..., :3]\n    import cv2\n    success, bgr = cam.read()\n    if not success: raise RuntimeError(\'Camera failed to return a frame\')\n    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n\n\ndef prep(rgb, input_info):\n    height,width = input_info[\'shape\'][1:3]\n    if len(input_info[\'shape\'])!=4 or input_info[\'shape\'][-1]!=3:\n        raise ValueError(\'Expected [1,height,width,3] RGB input; got \'+str(input_info[\'shape\']))\n    pil = Image.fromarray(np.asarray(rgb,dtype=np.uint8),mode=\'RGB\')\n    pil = ImageOps.fit(pil,(int(width),int(height)),method=Image.Resampling.BILINEAR)\n    x = np.asarray(pil,dtype=np.float32)[None,...]  # [0,255], no /255\n    dt = input_info[\'dtype\']\n    if np.issubdtype(dt,np.integer):\n        scale,zero = input_info[\'quantization\']\n        if scale <= 0: raise ValueError(\'Missing int8/uint8 input quantization scale\')\n        lim = np.iinfo(dt)\n        x = np.clip(np.round(x / scale + zero),lim.min,lim.max).astype(dt)\n    else:\n        x = x.astype(dt)\n    return x\n\n\ndef infer(interpreter, rgb):\n    inp=interpreter.get_input_details()[0]\n    out=interpreter.get_output_details()[0]\n    tensor=prep(rgb,inp)\n    t0=time.perf_counter()\n    interpreter.set_tensor(inp[\'index\'],tensor)\n    interpreter.invoke()\n    ms=1000*(time.perf_counter()-t0)\n    logits=interpreter.get_tensor(out[\'index\'])[0]\n    if np.issubdtype(out[\'dtype\'],np.integer):\n        scale,zp=out[\'quantization\']\n        logits=(logits.astype(np.float32)-zp)*scale\n    return np.asarray(logits,dtype=np.float32), ms\n\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'CaféGuard PR camera + TFLite on Raspberry Pi\')\n    ap.add_argument(\'--model\', required=True,type=Path)\n    ap.add_argument(\'--labels\',default=\'labels.json\',type=Path)\n    ap.add_argument(\'--camera\',choices=[\'auto\',\'pi\',\'usb\'],default=\'auto\')\n    ap.add_argument(\'--threads\',type=int,default=2)\n    ap.add_argument(\'--benchmark\',type=int,default=100,help=\'Run this many warmed-up inferences on the first live camera frame\')\n    ap.add_argument(\'--frames\',type=int,default=100,help=\'Number of live camera frames to classify; 0 is indefinite\')\n    ap.add_argument(\'--show\',action=\'store_true\',help=\'Show a GUI window (requires a desktop)\')\n    args=ap.parse_args()\n    if not args.model.is_file(): raise SystemExit(\'Missing TFLite file: \'+str(args.model))\n    labels=json.loads(args.labels.read_text())\n    interpreter, runtime = get_interpreter(args.model,args.threads)\n    print(\'Runtime:\',runtime,\'| Input:\',interpreter.get_input_details()[0][\'dtype\'])\n    cam,cam_type=open_camera(args.camera)\n    try:\n        frame=capture_rgb(cam,cam_type)\n        for _ in range(10): infer(interpreter,frame)\n        if args.benchmark > 0:\n            lat=[]\n            for _ in range(args.benchmark):\n                _,ms=infer(interpreter,frame)\n                lat.append(ms)\n            result={\'model\':args.model.name,\'samples\':len(lat),\'mean_ms\':float(np.mean(lat)),\n                    \'p95_ms\':float(np.percentile(lat,95)),\'median_ms\':float(np.median(lat)),\n                    \'inference_only_fps\':float(1000./np.mean(lat)),\n                    \'model_size_mib\':args.model.stat().st_size/(1024**2),\n                    \'camera\':cam_type, \'threads\':args.threads}\n            try:\n                import psutil\n                proc=psutil.Process()\n                result[\'process_ram_mib\']=proc.memory_info().rss/(1024**2)\n                result[\'system_cpu_percent_sample\']=psutil.cpu_percent(interval=0.2)\n            except ImportError:\n                pass\n            print(\'Pi benchmark (camera frame, inference-only):\',json.dumps(result,indent=2))\n            bench=Path(\'pi_benchmark_results.csv\')\n            existing=bench.is_file()\n            with bench.open(\'a\',newline=\'\') as f:\n                writer=csv.DictWriter(f,fieldnames=list(result))\n                if not existing: writer.writeheader()\n                writer.writerow(result)\n        end = None if args.frames==0 else args.frames\n        frames=0\n        start=time.perf_counter()\n        recent=deque(maxlen=30)\n        cv2=None\n        if args.show:\n            import cv2 as cv2\n        while end is None or frames < end:\n            rgb=capture_rgb(cam,cam_type)\n            probs,ms=infer(interpreter,rgb)\n            idx=int(np.argmax(probs))\n            cls=labels[idx] if idx < len(labels) else str(idx)\n            confidence=float(probs[idx])\n            recent.append(ms)\n            frames+=1\n            if frames%10==0 or frames==1:\n                print(f\'Frame {frames}: {cls} ({confidence:.1%}), inference {ms:.1f} ms\')\n            if args.show and cv2 is not None:\n                bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)\n                cv2.putText(bgr,f\'{cls} {confidence:.1%} | {ms:.0f} ms\',\n                    (10,35),cv2.FONT_HERSHEY_SIMPLEX,0.8,(20,220,20),2)\n                cv2.imshow(\'CafeGuard PR (press q to exit)\',bgr)\n                if cv2.waitKey(1)&0xff==ord(\'q\'): break\n        elapsed=time.perf_counter()-start\n        if frames:\n            print(f\'Processed {frames} live frames in {elapsed:.1f} sec; end-to-end FPS={frames/max(elapsed,1e-9):.2f}; mean recent inference={np.mean(recent):.1f}ms\')\n    finally:\n        if cam_type==\'pi\': cam.stop()\n        else: cam.release()\n        if args.show:\n            import cv2\n            cv2.destroyAllWindows()\n\n\nif __name__==\'__main__\':\n    main()\n'
(OUTPUT_DIR / "cafeguard_pi_camera.py").write_text(PI_SCRIPT)
print("Included Pi live-camera script in the output bundle")

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4))
axs[0].bar(results_df['model'],results_df['accuracy']*100)
axs[0].axhline(85,linestyle='--',color='gray',label='85% goal')
axs[0].set(ylabel='Held-out test accuracy (%)',title='Coffee-leaf accuracy',ylim=(0,100))
axs[0].tick_params(axis='x',rotation=20);axs[0].legend()
axs[1].bar(results_df['model'],results_df['size_mib'])
axs[1].set(ylabel='Model file size (MiB)',title='TFLite model size')
axs[1].tick_params(axis='x',rotation=20)
plt.tight_layout();plt.savefig(OUTPUT_DIR/'comparison.png',dpi=170);plt.show()

metadata={'project':'CaféGuard PR','source':source_description,'class_names':CLASS_NAMES,
          'selected_unique_images':dict(counts),'train':int(len(train_idx)),'val':int(len(val_idx)),
          'test':int(len(test_idx)),'image_size':IMAGE_SIZE,'pixel_convention':'RGB float [0,255]',
          'architecture':'MobileNetV3Small','qat_scope':'classifier head only; backbone quantized during conversion',
          'warnings':['A randomized image split is not a RoCoLe plant-grouped split.',
                      'Colab CPU timings cannot be reported as Raspberry Pi timings.',
                      'No Puerto Rican field data were tested in this notebook.']}
(OUTPUT_DIR/'experiment_metadata.json').write_text(json.dumps(metadata,indent=2))
readme="""CaféGuard PR deployment bundle\n\nModels:\n- cafeguard_fp32.tflite (float32 TFLite baseline)\n- cafeguard_int8_ptq.tflite (fully integer PTQ)\n- cafeguard_int8_head_qat.tflite (fully integer; only classifier head was QAT fine-tuned)\n\nInput: 224x224 RGB pixels in [0,255]. Pi script handles int8 quantization and output dequantization.\nlabels.json: [healthy, rust]. Rust excludes red-spider-mite-only cases.\n\nOn Raspberry Pi: install a compatible ai-edge-litert wheel + camera support; run cafeguard_pi_camera.py with --model <file> --benchmark 100.\nThe script reads a live Pi Camera (picamera2) or USB camera (OpenCV). Check actual FPS, CPU, and RAM separately.\n"""
(OUTPUT_DIR/'README.txt').write_text(readme)
from google.colab import files
bundle=Path('/content/CafeGuard_PR_Deployment.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.iterdir():
        if p.is_file() and p.name!='dataset_split.csv': z.write(p,p.name)
print('Prepared:',bundle)
files.download(str(bundle))

## 11. Final experiment checklist

- [ ] Confirm the final RoCoLe labels, exclusions and split by plant rather than individual photo (if plant identifiers available).
- [ ] Capture notebook graphs and save `colab_results.csv` after running.
- [ ] Record **FP32 versus PTQ versus head-QAT**, and correctly identify what was quantization-aware trained.
- [ ] Transfer the TFLite files, `labels.json`, and Pi inference script to the Raspberry Pi.
- [ ] Test a **live** Pi camera feed: RGB channels, 224×224 resolution, confidence display, stable loop.
- [ ] Benchmark **100+** single-frame inferences after warmup per model on the actual Pi: mean/p95 latency, FPS, CPU, RAM; energy only if measured.
- [ ] Make final poster/demo/video and explain whether domain shift to Puerto Rican farms could reduce accuracy.

**Accuracy, latency, size, and all other metrics must come from actual completed runs.**